# Lesson 2: The Property Ledger - Reading Data in Spark

---

## The Story

You've just joined a real estate analytics firm. The previous analytics team left behind data in **three different formats** scattered across the file system — CSVs, JSON files, and Parquet files. Before you can do any analysis, you need to load it all into Spark.

Think of it like inheriting a property ledger kept in different formats: some handwritten (CSV), some in a structured binder (JSON), and some in a compressed digital archive (Parquet). Your job is to read them all.

---

## What You'll Learn

- How to read **CSV** files with schema inference and explicit schemas
- How to read **JSON** files (including multi-line JSON)
- How to read **Parquet** files — the analytics champion
- The difference between **schema inference** and **explicit schema** definition
- How to use `show()`, `printSchema()`, and `count()` to inspect your data

---

## Datasets

| File | Format | Records | Description |
|------|--------|---------|-------------|
| `data/property_listings.csv` | CSV | 510 | Property listings with price, size, location |
| `data/agent_commissions.json` | JSON | 363 | Agent sales and commission records |
| `data/neighborhood_stats.parquet` | Parquet | 8 | Neighborhood demographic and quality stats |
| `data/mortgage_rates.csv` | CSV | 731 | Daily mortgage rate history |


In [1]:
# If running in a plain Python environment, install PySpark first:
# !pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    FloatType
)

print("Imports successful!")

Imports successful!


In [2]:
# Create a SparkSession - the entry point to all Spark functionality
# Think of it as opening the ledger book for the first time
spark = SparkSession.builder \
    .appName("ThePropertyLedger") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print("SparkSession created successfully!")

Spark version: 4.1.1
SparkSession created successfully!


## Reading CSV - The Most Common Format

CSV (Comma-Separated Values) is the most widely used data exchange format. It is human-readable but has no built-in schema — Spark needs to figure out (or be told) what each column's data type is.

### Schema Inference

When you set `inferSchema=True`, Spark reads the entire file once to detect column types. It is convenient but slow on large files — Spark essentially reads everything twice.

> **Property analogy:** Schema inference is like having an appraiser walk through a property and guess what everything is worth. It works, but it takes time and might not always be perfectly accurate.


In [3]:
# Read the property listings CSV with schema inference
listings_inferred = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/property_listings.csv")

print("=== Top 5 Property Listings (Inferred Schema) ===")
listings_inferred.show(5, truncate=False)

print("\n=== Schema (Inferred) ===")
listings_inferred.printSchema()

=== Top 5 Property Listings (Inferred Schema) ===


+-----------+-------------+-----------+--------+-------------+--------+---------+------+----------+----------+------------+--------+------------+--------+
|property_id|address      |city       |zip_code|property_type|bedrooms|bathrooms|sqft  |list_price|year_built|neighborhood|status  |listing_date|agent_id|
+-----------+-------------+-----------+--------+-------------+--------+---------+------+----------+----------+------------+--------+------------+--------+
|PROP-0001  |7370 River Rd|Parkview   |84038   |House        |5       |1.0      |NULL  |1268597.0 |1988.0    |Riverside   |Sold    |2025-12-24  |AGT-016 |
|PROP-0002  |960 Park Blvd|Parkview   |62393   |House        |NULL    |2.0      |4859.0|835503.0  |1955.0    |Oakwood     |NULL    |2025-10-07  |AGT-004 |
|PROP-0003  |5291 Hill Dr |Parkview   |76081   |House        |3       |1.0      |4882.0|487283.0  |2018.0    |Hillside    |Pending |NULL        |AGT-017 |
|PROP-0004  |5834 Oak Ave |Springfield|39430   |Condo        |3       

In [4]:
# Define an EXPLICIT schema - faster and more reliable than inferSchema
# This is the production-grade approach: you know your data, you define the types
listings_schema = StructType([
    StructField("property_id",   StringType(),  True),
    StructField("address",       StringType(),  True),
    StructField("city",          StringType(),  True),
    StructField("zip_code",      StringType(),  True),
    StructField("property_type", StringType(),  True),
    StructField("bedrooms",      IntegerType(), True),
    StructField("bathrooms",     FloatType(),   True),
    StructField("sqft",          IntegerType(), True),
    StructField("list_price",    DoubleType(),  True),
    StructField("year_built",    IntegerType(), True),
    StructField("neighborhood",  StringType(),  True),
    StructField("status",        StringType(),  True),
    StructField("listing_date",  StringType(),  True),
    StructField("agent_id",      StringType(),  True)
])

listings_df = spark.read \
    .option("header", True) \
    .schema(listings_schema) \
    .csv("data/property_listings.csv")

print("=== Top 5 Property Listings (Explicit Schema) ===")
listings_df.show(5)

print("\n=== Schema (Explicit - clean and precise) ===")
listings_df.printSchema()

print("\n[TIP] Explicit schema is preferred in production: faster, predictable, no surprises!")

=== Top 5 Property Listings (Explicit Schema) ===


+-----------+-------------+-----------+--------+-------------+--------+---------+----+----------+----------+------------+--------+------------+--------+
|property_id|      address|       city|zip_code|property_type|bedrooms|bathrooms|sqft|list_price|year_built|neighborhood|  status|listing_date|agent_id|
+-----------+-------------+-----------+--------+-------------+--------+---------+----+----------+----------+------------+--------+------------+--------+
|  PROP-0001|7370 River Rd|   Parkview|   84038|        House|       5|      1.0|NULL| 1268597.0|      NULL|   Riverside|    Sold|  2025-12-24| AGT-016|
|  PROP-0002|960 Park Blvd|   Parkview|   62393|        House|    NULL|      2.0|NULL|  835503.0|      NULL|     Oakwood|    NULL|  2025-10-07| AGT-004|
|  PROP-0003| 5291 Hill Dr|   Parkview|   76081|        House|       3|      1.0|NULL|  487283.0|      NULL|    Hillside| Pending|        NULL| AGT-017|
|  PROP-0004| 5834 Oak Ave|Springfield|   39430|        Condo|       3|      2.5|N

## Reading JSON - Semi-Structured Data

JSON is more expressive than CSV — it supports nested structures and arrays. Spark can read JSON files where each line is a JSON object (the default) or files where the entire document is a single JSON array (`multiLine` mode).

### JSON vs CSV

- JSON is **self-describing** — field names are embedded in the data
- JSON supports **nested objects** and **arrays** natively
- JSON files are typically **larger** than equivalent CSVs

> **Property analogy:** JSON is like a property disclosure form — it has named sections, can include nested details (`appliances: {washer: true, dryer: true}`), but takes more space than a simple spreadsheet row.


In [5]:
# Read agent commissions from JSON
# multiLine=True handles JSON files where the entire content is one JSON array
commissions_df = spark.read \
    .option("multiLine", True) \
    .json("data/agent_commissions.json")

print("=== Agent Commissions (JSON) ===")
commissions_df.show(5)

print("\n=== Schema (auto-detected from JSON) ===")
commissions_df.printSchema()

print(f"\nTotal commission records: {commissions_df.count()}")

=== Agent Commissions (JSON) ===


+--------+-----------------+---------------+-----------+----------+----------+
|agent_id|commission_amount|commission_rate|property_id| sale_date|sale_price|
+--------+-----------------+---------------+-----------+----------+----------+
| AGT-001|          4292.66|           0.02|  PROP-0405|2026-01-24|    214633|
| AGT-001|          9702.75|           0.03|  PROP-0438|2025-12-04|    323425|
| AGT-001|          9903.64|           0.02|  PROP-0035|2026-02-26|    495182|
| AGT-001|         20010.06|           0.03|  PROP-0234|2026-01-03|    667002|
| AGT-001|         10036.65|          0.025|  PROP-0086|2025-09-24|    401466|
+--------+-----------------+---------------+-----------+----------+----------+
only showing top 5 rows

=== Schema (auto-detected from JSON) ===
root
 |-- agent_id: string (nullable = true)
 |-- commission_amount: double (nullable = true)
 |-- commission_rate: double (nullable = true)
 |-- property_id: string (nullable = true)
 |-- sale_date: string (nullable = true


Total commission records: 363


## Reading Parquet - The Analytics Champion

Parquet is a **columnar storage format** designed specifically for analytics workloads. It is the preferred format in the modern data stack (used by Spark, Hive, Presto, Databricks, and more).

### Why Parquet Wins for Analytics

1. **Columnar storage** — if you only query 3 of 20 columns, Spark only reads those 3 columns from disk
2. **Schema embedded** — no need for `inferSchema` or explicit schema; the schema is stored in the file footer
3. **Built-in compression** — typically 4-10x smaller than equivalent CSV
4. **Predicate pushdown** — Spark can skip entire row groups that do not match your filter

> **Property analogy:** Parquet is like a fully digitized, indexed property archive — instantly searchable, compact, and you always know exactly what is inside without opening every folder.


In [6]:
# Read neighborhood stats from Parquet - the simplest read of all!
# No options needed: schema is embedded in the file
neighborhoods_df = spark.read.parquet("data/neighborhood_stats.parquet")

print("=== Neighborhood Statistics (Parquet) ===")
neighborhoods_df.show(truncate=False)

print("\n=== Schema (embedded in Parquet file - no inference needed) ===")
neighborhoods_df.printSchema()

print("\n[Notice] The schema is already perfectly typed - no inferSchema, no explicit definition needed!")

=== Neighborhood Statistics (Parquet) ===


+------------+-------------+-------------+-------------------+----------+----------------+-----------+
|neighborhood|median_income|school_rating|crime_rate_per_1000|population|avg_commute_time|parks_count|
+------------+-------------+-------------+-------------------+----------+----------------+-----------+
|Downtown    |145967       |7.6          |16.51              |23907     |23              |4          |
|Riverside   |138989       |8.3          |24.01              |11159     |16              |9          |
|Hillside    |85079        |9.8          |22.18              |34158     |31              |8          |
|Oakwood     |131556       |8.1          |9.57               |25293     |41              |3          |
|Parkview    |99323        |9.3          |9.4                |38932     |43              |13         |
|Seaside     |95320        |8.2          |9.02               |38387     |19              |8          |
|Midtown     |142098       |6.2          |23.45              |45750     |

In [7]:
# Read the mortgage rates CSV - another CSV example with date data
mortgage_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/mortgage_rates.csv")

print("=== Mortgage Rates (first 5 rows) ===")
mortgage_df.show(5)

print("\n=== Schema ===")
mortgage_df.printSchema()

print(f"\nTotal mortgage rate records: {mortgage_df.count()}")

=== Mortgage Rates (first 5 rows) ===
+----------+---------------+---------------+------------+
|      date|rate_30yr_fixed|rate_15yr_fixed|rate_5_1_arm|
+----------+---------------+---------------+------------+
|2023-01-01|          6.711|          6.211|       5.911|
|2023-01-02|          6.756|          6.256|       5.956|
|2023-01-03|           6.81|           6.31|        6.01|
|2023-01-04|          6.768|          6.268|       5.968|
|2023-01-05|          6.467|          5.967|       5.667|
+----------+---------------+---------------+------------+
only showing top 5 rows

=== Schema ===
root
 |-- date: date (nullable = true)
 |-- rate_30yr_fixed: double (nullable = true)
 |-- rate_15yr_fixed: double (nullable = true)
 |-- rate_5_1_arm: double (nullable = true)




Total mortgage rate records: 731


## Comparing File Formats

Here is a quick reference guide for choosing the right format:

| Feature | CSV | JSON | Parquet |
|---------|-----|------|---------|
| **Schema support** | None (must infer or define) | Self-describing field names | Full schema embedded in file |
| **Compression** | None by default | None by default | Built-in (Snappy / GZIP) |
| **Read speed** | Slow (row-based scan) | Slow (parse overhead) | Fast (columnar, skip unused cols) |
| **Human readable** | Yes | Yes | No (binary format) |
| **Nested data** | No | Yes | Yes |
| **Best for** | Data exchange, exports | APIs, configs, logs | Analytics, data lakes |
| **Spark preference** | Works, but slower | Works, more memory | Preferred for production |

> **Rule of thumb:** Use Parquet for everything stored long-term in your data lake. Use CSV/JSON only for raw ingestion or when interoperability with external tools is required.


In [8]:
# Summary: record counts for all 4 datasets
print("=" * 55)
print("   PROPERTY ANALYTICS - DATASET INVENTORY SUMMARY")
print("=" * 55)

datasets = [
    ("Property Listings",  "CSV",     listings_df.count()),
    ("Agent Commissions",  "JSON",    commissions_df.count()),
    ("Neighborhood Stats", "Parquet", neighborhoods_df.count()),
    ("Mortgage Rates",     "CSV",     mortgage_df.count()),
]

print(f"  {'Dataset':<25} {'Format':<10} {'Records':>8}")
print("-" * 55)
total = 0
for name, fmt, cnt in datasets:
    print(f"  {name:<25} {fmt:<10} {cnt:>8,}")
    total += cnt

print("-" * 55)
print(f"  {'TOTAL':<25} {'':10} {total:>8,}")
print("=" * 55)

   PROPERTY ANALYTICS - DATASET INVENTORY SUMMARY


  Dataset                   Format      Records
-------------------------------------------------------
  Property Listings         CSV             510
  Agent Commissions         JSON            363
  Neighborhood Stats        Parquet           8
  Mortgage Rates            CSV             731
-------------------------------------------------------
  TOTAL                                   1,612


## Lesson 2 Wrap-Up

You have successfully loaded all four datasets that power this course. Here is what you have mastered:

### Key Takeaways

- **`spark.read.csv()`** — reads CSV files; use `option("header", True)` and either `inferSchema` or an explicit `StructType`
- **`spark.read.json()`** — reads JSON files; use `option("multiLine", True)` for single-document JSON arrays
- **`spark.read.parquet()`** — simplest of all; schema is embedded, no extra options needed
- **Explicit schema > inferSchema** in production: faster, type-safe, no surprises
- **`show()`**, **`printSchema()`**, and **`count()`** are your go-to DataFrame inspection tools

### The Spark Read API Pattern

```python
df = spark.read \
    .option("key", "value") \
    .schema(my_schema) \
    .csv("path/to/file.csv")
```

---

**Next: Lesson 3 - Touring the Listings**

Now that the data is loaded, it is time to walk through it — selecting columns, filtering rows, and transforming values to find exactly what you are looking for.
